In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import pandas as pd

# Project root
ROOT = Path(PROJECT_ROOT)

if not ROOT.exists():
    raise FileNotFoundError(f"Project folder not found: {ROOT}")

# Terms likely to appear in objective-weight columns
keywords = (
    "weight",
    "lambda",
    "profile",
    "privacy",
    "interpret",
    "parsim",
    "cardinality",
)

records = []
errors = []

# Search every CSV in the complete project
for csv_path in ROOT.rglob("*.csv"):
    try:
        columns = list(pd.read_csv(csv_path, nrows=0).columns)

        matching_columns = [
            column
            for column in columns
            if any(keyword in column.lower() for keyword in keywords)
        ]

        if matching_columns:
            records.append(
                {
                    "file_name": csv_path.name,
                    "relative_path": str(csv_path.relative_to(ROOT)),
                    "matching_columns": " | ".join(matching_columns),
                    "all_columns": " | ".join(columns),
                }
            )

    except Exception as exc:
        errors.append(
            {
                "file": str(csv_path),
                "error": str(exc),
            }
        )

weight_column_audit = pd.DataFrame(records)

if weight_column_audit.empty:
    print("No likely weight-related columns were found in the CSV files.")
    print("The next search will examine Python files, notebooks, and configuration files.")
else:
    weight_column_audit = weight_column_audit.sort_values(
        ["relative_path", "file_name"]
    ).reset_index(drop=True)

    display(weight_column_audit)

    output_path = (
        ROOT
        / "05_Results"
        / "methodology_weight_column_audit.csv"
    )

    weight_column_audit.to_csv(output_path, index=False)

    print("\nAudit saved to:")
    print(output_path)

if errors:
    print(f"\nCSV files that could not be inspected: {len(errors)}")
    display(pd.DataFrame(errors))

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import pandas as pd

ROOT = Path(PROJECT_ROOT)

benchmark_file = (
    ROOT
    / "05_Results"
    / "corrected_v2_algorithm_comparison_long.csv"
)

if not benchmark_file.exists():
    raise FileNotFoundError(f"File not found: {benchmark_file}")

df = pd.read_csv(benchmark_file)

print("Source file:")
print(benchmark_file)

print("\nAll columns containing 'weight' or 'lambda':")
relevant_columns = [
    column
    for column in df.columns
    if "weight" in column.lower() or "lambda" in column.lower()
]

for column in relevant_columns:
    print(f"  - {column}")

profile_column = "weight_profile"

if profile_column not in df.columns:
    raise KeyError(
        f"Expected column '{profile_column}' was not found."
    )

lambda_columns = [
    column
    for column in df.columns
    if column.lower().startswith("lambda_")
]

if len(lambda_columns) != 3:
    raise ValueError(
        "Expected exactly three lambda columns, but found: "
        f"{lambda_columns}"
    )

weight_profiles = (
    df[[profile_column] + lambda_columns]
    .drop_duplicates()
    .sort_values(profile_column)
    .reset_index(drop=True)
)

print("\nUnique objective-weight profiles:")
display(weight_profiles)

output_file = (
    ROOT
    / "05_Results"
    / "methodology_exact_weight_profiles.csv"
)

weight_profiles.to_csv(output_file, index=False)

print("\nExact weight profiles saved to:")
print(output_file)

In [ ]:
consistency_rows = []

for profile_name, group in df.groupby("weight_profile"):
    unique_triples = (
        group[lambda_columns]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    consistency_rows.append(
        {
            "weight_profile": profile_name,
            "number_of_unique_weight_triples": len(unique_triples),
            "row_count": len(group),
            "consistent": len(unique_triples) == 1,
        }
    )

consistency_audit = pd.DataFrame(consistency_rows)

display(consistency_audit)

if consistency_audit["consistent"].all():
    print("PASS: Every weight profile uses exactly one numerical triple.")
else:
    print("WARNING: At least one profile contains multiple numerical triples.")

consistency_file = (
    ROOT
    / "05_Results"
    / "methodology_weight_profile_consistency_audit.csv"
)

consistency_audit.to_csv(consistency_file, index=False)

print("\nConsistency audit saved to:")
print(consistency_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import re
import pandas as pd

ROOT = Path(PROJECT_ROOT)

# Source and configuration formats that may contain numerical constants
extensions = {
    ".py",
    ".java",
    ".sarl",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".ini",
    ".cfg",
    ".txt",
    ".md",
}

# Search expressions related to epsilon and floating-point tolerance
patterns = {
    "epsilon": re.compile(
        r"\b(epsilon|eps|EPSILON|EPS)\b|1e-\d+",
        re.IGNORECASE,
    ),
    "tolerance": re.compile(
        r"\b(tolerance|tol|atol|rtol|isclose|allclose|"
        r"math\.abs|Math\.abs|np\.abs|numpy\.abs)\b",
        re.IGNORECASE,
    ),
    "comparison": re.compile(
        r"(objective.*equal|equal.*objective|"
        r"numerical.*equal|floating.*point|"
        r"gap.*zero|threshold.*tolerance)",
        re.IGNORECASE,
    ),
}

records = []
read_errors = []

for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()

        for line_number, line in enumerate(lines, start=1):
            matched_categories = [
                category
                for category, pattern in patterns.items()
                if pattern.search(line)
            ]

            if matched_categories:
                records.append(
                    {
                        "relative_path": str(path.relative_to(ROOT)),
                        "file_type": path.suffix.lower(),
                        "line_number": line_number,
                        "categories": " | ".join(matched_categories),
                        "line_text": line.strip(),
                    }
                )

    except Exception as exc:
        read_errors.append(
            {
                "relative_path": str(path.relative_to(ROOT)),
                "error": str(exc),
            }
        )

numerical_constant_search = pd.DataFrame(records)

if numerical_constant_search.empty:
    print("No epsilon or tolerance references were found.")
else:
    numerical_constant_search = (
        numerical_constant_search
        .sort_values(
            ["relative_path", "line_number"]
        )
        .reset_index(drop=True)
    )

    pd.set_option("display.max_colwidth", 220)
    display(numerical_constant_search)

    output_file = (
        ROOT
        / "05_Results"
        / "methodology_epsilon_tolerance_source_search.csv"
    )

    numerical_constant_search.to_csv(output_file, index=False)

    print("\nSearch results saved to:")
    print(output_file)

if read_errors:
    print("\nFiles that could not be read:")
    display(pd.DataFrame(read_errors))

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path(PROJECT_ROOT)

notebook_patterns = {
    "epsilon": re.compile(
        r"\b(epsilon|eps)\b|1e-\d+",
        re.IGNORECASE,
    ),
    "tolerance": re.compile(
        r"\b(tolerance|tol|atol|rtol|isclose|allclose)\b",
        re.IGNORECASE,
    ),
    "comparison": re.compile(
        r"(objective.*equal|equal.*objective|"
        r"numerical.*equal|floating.*point)",
        re.IGNORECASE,
    ),
}

notebook_records = []
notebook_errors = []

for notebook_path in ROOT.rglob("*.ipynb"):
    try:
        with notebook_path.open(
            "r",
            encoding="utf-8",
            errors="ignore",
        ) as handle:
            notebook = json.load(handle)

        code_cell_index = 0

        for cell in notebook.get("cells", []):
            if cell.get("cell_type") != "code":
                continue

            code_cell_index += 1
            source = "".join(cell.get("source", []))

            for source_line_number, line in enumerate(
                source.splitlines(),
                start=1,
            ):
                matched_categories = [
                    category
                    for category, pattern in notebook_patterns.items()
                    if pattern.search(line)
                ]

                if matched_categories:
                    notebook_records.append(
                        {
                            "relative_path": str(
                                notebook_path.relative_to(ROOT)
                            ),
                            "code_cell_index": code_cell_index,
                            "source_line_number": source_line_number,
                            "categories": " | ".join(
                                matched_categories
                            ),
                            "line_text": line.strip(),
                        }
                    )

    except Exception as exc:
        notebook_errors.append(
            {
                "relative_path": str(
                    notebook_path.relative_to(ROOT)
                ),
                "error": str(exc),
            }
        )

notebook_constant_search = pd.DataFrame(notebook_records)

if notebook_constant_search.empty:
    print("No epsilon or tolerance references were found in notebooks.")
else:
    notebook_constant_search = (
        notebook_constant_search
        .sort_values(
            [
                "relative_path",
                "code_cell_index",
                "source_line_number",
            ]
        )
        .reset_index(drop=True)
    )

    pd.set_option("display.max_colwidth", 220)
    display(notebook_constant_search)

    output_file = (
        ROOT
        / "05_Results"
        / "methodology_epsilon_tolerance_notebook_search.csv"
    )

    notebook_constant_search.to_csv(output_file, index=False)

    print("\nNotebook search saved to:")
    print(output_file)

if notebook_errors:
    print("\nNotebooks that could not be inspected:")
    display(pd.DataFrame(notebook_errors))

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import re
import pandas as pd

ROOT = Path(PROJECT_ROOT)
SOURCE = ROOT / "03_Python_Optimization" / "mst_core.py"

if not SOURCE.exists():
    raise FileNotFoundError(f"Final module not found: {SOURCE}")

lines = SOURCE.read_text(
    encoding="utf-8",
    errors="ignore",
).splitlines()

# Terms identifying the relevant functions and numerical comparisons
target_patterns = [
    r"def\s+exact_mst",
    r"def\s+compute_optimality_gap",
    r"def\s+subset_metrics",
    r"def\s+greedy_mst",
    r"def\s+progressive_mst",
    r"isclose",
    r"allclose",
    r"1e-\d+",
    r"abs\s*\(",
    r"objective",
    r"threshold",
    r"best_",
    r"utility\s*[<>]=?",
    r"tau\s*[<>]=?",
]

combined_pattern = re.compile(
    "|".join(target_patterns),
    re.IGNORECASE,
)

matches = []

for index, line in enumerate(lines):
    if combined_pattern.search(line):
        start = max(0, index - 4)
        end = min(len(lines), index + 5)

        context = "\n".join(
            f"{line_no + 1}: {lines[line_no]}"
            for line_no in range(start, end)
        )

        matches.append(
            {
                "matched_line": index + 1,
                "line_text": line.strip(),
                "context": context,
            }
        )

comparison_context = pd.DataFrame(matches)

pd.set_option("display.max_colwidth", 500)
display(comparison_context)

output_file = (
    ROOT
    / "05_Results"
    / "methodology_exact_comparison_context.csv"
)

comparison_context.to_csv(output_file, index=False)

print("\nComparison context saved to:")
print(output_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import re
import pandas as pd

ROOT = Path(PROJECT_ROOT)

patterns = re.compile(
    r"""
    isclose|
    allclose|
    atol|
    rtol|
    tolerance|
    \btol\b|
    1e-\d+|
    exact[_\s-]*recovery|
    objective[_\s-]*reconstruction|
    optimality[_\s-]*gap|
    abs\s*\([^)]*(objective|gap|utility)
    """,
    re.IGNORECASE | re.VERBOSE,
)

records = []

for path in ROOT.rglob("*.py"):
    try:
        lines = path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()

        for line_number, line in enumerate(lines, start=1):
            if patterns.search(line):
                records.append(
                    {
                        "relative_path": str(path.relative_to(ROOT)),
                        "line_number": line_number,
                        "line_text": line.strip(),
                    }
                )

    except Exception as exc:
        records.append(
            {
                "relative_path": str(path.relative_to(ROOT)),
                "line_number": None,
                "line_text": f"READ ERROR: {exc}",
            }
        )

tolerance_audit = pd.DataFrame(records)

if tolerance_audit.empty:
    print("No additional tolerance-related lines were found.")
else:
    tolerance_audit = tolerance_audit.sort_values(
        ["relative_path", "line_number"]
    ).reset_index(drop=True)

    pd.set_option("display.max_colwidth", 300)
    display(tolerance_audit)

    output_file = (
        ROOT
        / "05_Results"
        / "methodology_all_python_tolerance_audit.csv"
    )

    tolerance_audit.to_csv(output_file, index=False)

    print("\nAudit saved to:")
    print(output_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path

source = Path(PROJECT_ROOT) / "03_Python_Optimization" / "mst_core.py"

lines = source.read_text(
    encoding="utf-8",
    errors="ignore",
).splitlines()

for line_number in range(929, min(1005, len(lines)) + 1):
    print(f"{line_number}: {lines[line_number - 1]}")

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import pandas as pd

ROOT = Path(PROJECT_ROOT)

SOURCE = (
    ROOT
    / "05_Results"
    / "corrected_v2_algorithm_comparison_long.csv"
)

if not SOURCE.exists():
    raise FileNotFoundError(f"Benchmark file not found: {SOURCE}")

df = pd.read_csv(SOURCE)

keywords = (
    "gap",
    "recovery",
    "recover",
    "exact",
    "greedy",
    "objective",
    "disclosure_cost",
    "optimal",
)

relevant_columns = [
    column
    for column in df.columns
    if any(keyword in column.lower() for keyword in keywords)
]

print("Relevant benchmark columns:\n")

for column in relevant_columns:
    print(f"- {column}")

print("\nColumn types and example values:\n")

summary_rows = []

for column in relevant_columns:
    non_null = df[column].dropna()

    summary_rows.append(
        {
            "column": column,
            "dtype": str(df[column].dtype),
            "non_null_rows": int(non_null.shape[0]),
            "unique_values": int(non_null.nunique()),
            "minimum": (
                non_null.min()
                if pd.api.types.is_numeric_dtype(non_null)
                else None
            ),
            "maximum": (
                non_null.max()
                if pd.api.types.is_numeric_dtype(non_null)
                else None
            ),
            "example_values": " | ".join(
                map(str, non_null.head(5).tolist())
            ),
        }
    )

column_audit = pd.DataFrame(summary_rows)

pd.set_option("display.max_colwidth", 250)
display(column_audit)

output_file = (
    ROOT
    / "05_Results"
    / "methodology_gap_recovery_column_audit.csv"
)

column_audit.to_csv(output_file, index=False)

print("\nAudit saved to:")
print(output_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(PROJECT_ROOT)

SOURCE = (
    ROOT
    / "05_Results"
    / "corrected_v2_algorithm_comparison_long.csv"
)

df = pd.read_csv(SOURCE)

# ---------------------------------------------------------
# 1. Identify columns that distinguish algorithm variants
# ---------------------------------------------------------

algorithm_identifier_columns = [
    column
    for column in df.columns
    if any(
        term in column.lower()
        for term in (
            "algorithm",
            "method",
            "solver",
            "variant",
            "heuristic",
        )
    )
]

print("Possible algorithm-identification columns:\n")

for column in algorithm_identifier_columns:
    print(f"- {column}")

print("\nUnique values in low-cardinality identifier columns:\n")

identifier_summaries = []

for column in algorithm_identifier_columns:
    unique_count = df[column].nunique(dropna=True)

    if unique_count <= 30:
        values = sorted(
            map(str, df[column].dropna().unique().tolist())
        )

        identifier_summaries.append(
            {
                "column": column,
                "unique_count": unique_count,
                "unique_values": " | ".join(values),
            }
        )

identifier_table = pd.DataFrame(identifier_summaries)
display(identifier_table)

# ---------------------------------------------------------
# 2. Restrict to rows with defined finite comparison values
# ---------------------------------------------------------

eligible = df[
    df["optimality_gap_percent"].notna()
    & df["algorithm_disclosure_cost"].notna()
    & df["exact_disclosure_cost"].notna()
    & np.isfinite(df["algorithm_disclosure_cost"])
    & np.isfinite(df["exact_disclosure_cost"])
].copy()

eligible["absolute_objective_difference"] = (
    eligible["algorithm_disclosure_cost"]
    - eligible["exact_disclosure_cost"]
)

eligible["direct_cost_equality"] = (
    eligible["algorithm_disclosure_cost"]
    == eligible["exact_disclosure_cost"]
)

eligible["direct_zero_gap"] = (
    eligible["optimality_gap_percent"]
    == 0.0
)

# Candidate numerical definitions for recovery
for tolerance in (1e-12, 1e-10, 1e-9, 1e-8, 1e-6):
    label = f"abs_diff_le_{tolerance:.0e}"

    eligible[label] = (
        eligible["absolute_objective_difference"].abs()
        <= tolerance
    )

# ---------------------------------------------------------
# 3. Compare stored recovery flag with candidate rules
# ---------------------------------------------------------

candidate_rules = [
    "direct_cost_equality",
    "direct_zero_gap",
    "abs_diff_le_1e-12",
    "abs_diff_le_1e-10",
    "abs_diff_le_1e-09",
    "abs_diff_le_1e-08",
    "abs_diff_le_1e-06",
]

comparison_rows = []

stored_flag = (
    eligible["algorithm_optimal_cost_numeric"]
    .astype(int)
    .astype(bool)
)

for rule in candidate_rules:
    predicted_flag = eligible[rule].astype(bool)

    comparison_rows.append(
        {
            "candidate_rule": rule,
            "eligible_rows": len(eligible),
            "stored_positive_rows": int(stored_flag.sum()),
            "predicted_positive_rows": int(predicted_flag.sum()),
            "mismatch_count": int(
                (stored_flag != predicted_flag).sum()
            ),
        }
    )

recovery_rule_comparison = pd.DataFrame(comparison_rows)

print("\nStored recovery flag compared with candidate definitions:\n")
display(recovery_rule_comparison)

# ---------------------------------------------------------
# 4. Examine the boundary around the stored recovery flag
# ---------------------------------------------------------

recovered = eligible[
    eligible["algorithm_optimal_cost_numeric"] == 1
]

not_recovered = eligible[
    eligible["algorithm_optimal_cost_numeric"] == 0
]

boundary_summary = pd.DataFrame(
    [
        {
            "group": "stored_recovery_1",
            "rows": len(recovered),
            "minimum_gap_percent": recovered[
                "optimality_gap_percent"
            ].min(),
            "maximum_gap_percent": recovered[
                "optimality_gap_percent"
            ].max(),
            "minimum_absolute_difference": recovered[
                "absolute_objective_difference"
            ].abs().min(),
            "maximum_absolute_difference": recovered[
                "absolute_objective_difference"
            ].abs().max(),
        },
        {
            "group": "stored_recovery_0",
            "rows": len(not_recovered),
            "minimum_gap_percent": not_recovered[
                "optimality_gap_percent"
            ].min(),
            "maximum_gap_percent": not_recovered[
                "optimality_gap_percent"
            ].max(),
            "minimum_absolute_difference": not_recovered[
                "absolute_objective_difference"
            ].abs().min(),
            "maximum_absolute_difference": not_recovered[
                "absolute_objective_difference"
            ].abs().max(),
        },
    ]
)

print("\nRecovery-boundary summary:\n")
display(boundary_summary)

# ---------------------------------------------------------
# 5. Summarize gaps by algorithm variant
# ---------------------------------------------------------

group_columns = [
    column
    for column in algorithm_identifier_columns
    if df[column].nunique(dropna=True) <= 30
]

algorithm_gap_summaries = []

for group_column in group_columns:
    grouped = (
        eligible
        .groupby(group_column, dropna=False)
        .agg(
            rows=("optimality_gap_percent", "size"),
            mean_gap_percent=("optimality_gap_percent", "mean"),
            median_gap_percent=("optimality_gap_percent", "median"),
            maximum_gap_percent=("optimality_gap_percent", "max"),
            optimal_cost_recovery_percent=(
                "algorithm_optimal_cost_numeric",
                lambda values: 100.0 * values.mean(),
            ),
            large_gap_percent=(
                "large_gap_numeric",
                lambda values: 100.0 * values.mean(),
            ),
        )
        .reset_index()
    )

    grouped.insert(0, "grouping_column", group_column)
    algorithm_gap_summaries.append(grouped)

if algorithm_gap_summaries:
    algorithm_gap_summary = pd.concat(
        algorithm_gap_summaries,
        ignore_index=True,
    )

    print("\nGap summaries by possible algorithm identifier:\n")
    display(algorithm_gap_summary)
else:
    algorithm_gap_summary = pd.DataFrame()
    print("No suitable algorithm grouping column was found.")

# ---------------------------------------------------------
# 6. Save the audits
# ---------------------------------------------------------

recovery_rule_file = (
    ROOT
    / "05_Results"
    / "methodology_recovery_rule_comparison.csv"
)

boundary_file = (
    ROOT
    / "05_Results"
    / "methodology_recovery_boundary_summary.csv"
)

algorithm_file = (
    ROOT
    / "05_Results"
    / "methodology_algorithm_variant_gap_summary.csv"
)

recovery_rule_comparison.to_csv(
    recovery_rule_file,
    index=False,
)

boundary_summary.to_csv(
    boundary_file,
    index=False,
)

if not algorithm_gap_summary.empty:
    algorithm_gap_summary.to_csv(
        algorithm_file,
        index=False,
    )

print("\nSaved:")
print(recovery_rule_file)
print(boundary_file)

if not algorithm_gap_summary.empty:
    print(algorithm_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import json
import pandas as pd

ROOT = Path(PROJECT_ROOT)

search_terms = (
    "algorithm_optimal_cost_numeric",
    "optimality_gap_percent",
    "large_gap_numeric",
)

records = []

# Search Python files
for path in ROOT.rglob("*.py"):
    lines = path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    for line_number, line in enumerate(lines, start=1):
        if any(term in line for term in search_terms):
            start = max(0, line_number - 5)
            end = min(len(lines), line_number + 4)

            context = "\n".join(
                f"{number + 1}: {lines[number]}"
                for number in range(start, end)
            )

            records.append(
                {
                    "source_type": "python",
                    "relative_path": str(path.relative_to(ROOT)),
                    "cell_or_line": line_number,
                    "matched_text": line.strip(),
                    "context": context,
                }
            )

# Search Jupyter notebooks
for path in ROOT.rglob("*.ipynb"):
    try:
        notebook = json.loads(
            path.read_text(
                encoding="utf-8",
                errors="ignore",
            )
        )

        code_cell_number = 0

        for cell in notebook.get("cells", []):
            if cell.get("cell_type") != "code":
                continue

            code_cell_number += 1
            source = "".join(cell.get("source", []))

            if any(term in source for term in search_terms):
                records.append(
                    {
                        "source_type": "notebook",
                        "relative_path": str(path.relative_to(ROOT)),
                        "cell_or_line": code_cell_number,
                        "matched_text": "One or more target fields found",
                        "context": source,
                    }
                )

    except Exception:
        pass

definition_search = pd.DataFrame(records)

pd.set_option("display.max_colwidth", 600)

if definition_search.empty:
    print("No source definitions were found.")
else:
    display(definition_search)

    output_file = (
        ROOT
        / "05_Results"
        / "methodology_gap_flag_definition_search.csv"
    )

    definition_search.to_csv(output_file, index=False)

    print("\nSaved:")
    print(output_file)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(PROJECT_ROOT)

SEARCH_FOLDERS = [
    ROOT / "05_Results",
    ROOT / "07_Tables",
]

records = []
errors = []

for folder in SEARCH_FOLDERS:
    if not folder.exists():
        continue

    for path in folder.rglob("*.csv"):
        try:
            df = pd.read_csv(path)

            if "optimality_gap_percent" not in df.columns:
                continue

            gap = pd.to_numeric(
                df["optimality_gap_percent"],
                errors="coerce",
            )

            finite_gap = gap[np.isfinite(gap)]

            algorithm_values = ""

            if "algorithm" in df.columns:
                algorithm_values = " | ".join(
                    sorted(
                        map(
                            str,
                            df["algorithm"]
                            .dropna()
                            .unique()
                            .tolist(),
                        )
                    )
                )

            benchmark_versions = ""

            version_columns = [
                column
                for column in df.columns
                if "version" in column.lower()
            ]

            if version_columns:
                version_parts = []

                for column in version_columns:
                    values = (
                        df[column]
                        .dropna()
                        .astype(str)
                        .unique()
                        .tolist()
                    )

                    if len(values) <= 10:
                        version_parts.append(
                            f"{column}="
                            + "|".join(sorted(values))
                        )

                benchmark_versions = "; ".join(version_parts)

            records.append(
                {
                    "relative_path": str(path.relative_to(ROOT)),
                    "total_rows": len(df),
                    "defined_gap_rows": int(gap.notna().sum()),
                    "finite_gap_rows": int(len(finite_gap)),
                    "mean_gap_percent": (
                        float(finite_gap.mean())
                        if len(finite_gap)
                        else np.nan
                    ),
                    "median_gap_percent": (
                        float(finite_gap.median())
                        if len(finite_gap)
                        else np.nan
                    ),
                    "maximum_gap_percent": (
                        float(finite_gap.max())
                        if len(finite_gap)
                        else np.nan
                    ),
                    "minimum_gap_percent": (
                        float(finite_gap.min())
                        if len(finite_gap)
                        else np.nan
                    ),
                    "algorithm_values": algorithm_values,
                    "benchmark_versions": benchmark_versions,
                    "has_failure_mechanism": (
                        "failure_mechanism" in df.columns
                    ),
                    "has_matched_group_id": (
                        "matched_group_id" in df.columns
                    ),
                    "has_exact_recovery_flag": (
                        "algorithm_optimal_cost_numeric"
                        in df.columns
                        or "exact_cost_recovery_numeric"
                        in df.columns
                    ),
                }
            )

        except Exception as exc:
            errors.append(
                {
                    "relative_path": str(path.relative_to(ROOT)),
                    "error": str(exc),
                }
            )

dataset_audit = pd.DataFrame(records)

if dataset_audit.empty:
    print("No CSV containing optimality_gap_percent was found.")
else:
    dataset_audit = (
        dataset_audit
        .sort_values(
            [
                "total_rows",
                "relative_path",
            ],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    pd.set_option("display.max_colwidth", 300)
    display(dataset_audit)

    output_file = (
        ROOT
        / "05_Results"
        / "methodology_optimality_gap_dataset_audit.csv"
    )

    dataset_audit.to_csv(output_file, index=False)

    print("\nAudit saved to:")
    print(output_file)

if errors:
    print("\nFiles that could not be inspected:")
    display(pd.DataFrame(errors))

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path
import pandas as pd

ROOT = Path(PROJECT_ROOT)
RESULTS = ROOT / "05_Results"

files = {
    "metadata": RESULTS / "corrected_generator_metadata.csv",
    "validation_raw": RESULTS / "corrected_generator_validation_raw.csv",
    "validation_summary": RESULTS / "corrected_generator_validation_summary.csv",
}

for name, path in files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} file: {path}")

    df = pd.read_csv(path, low_memory=False)

    print("\n" + "=" * 90)
    print(name.upper())
    print(path)
    print("=" * 90)

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    for column in df.columns:
        print(f"- {column}")

    print("\nFull contents:")

    if len(df) <= 100:
        display(df)
    else:
        display(df.head(20))

        print("\nUnique-value and range audit:")

        audit_rows = []

        for column in df.columns:
            series = df[column]
            non_null = series.dropna()

            row = {
                "column": column,
                "dtype": str(series.dtype),
                "non_null_rows": len(non_null),
                "unique_values": non_null.nunique(),
            }

            if pd.api.types.is_numeric_dtype(series):
                row["minimum"] = non_null.min()
                row["maximum"] = non_null.max()
                row["mean"] = non_null.mean()
            else:
                values = non_null.astype(str).unique()

                row["example_or_unique_values"] = " | ".join(
                    sorted(values[:20])
                )

            audit_rows.append(row)

        display(pd.DataFrame(audit_rows))

    output_path = (
        RESULTS
        / f"methodology_{name}_inspection.csv"
    )

    df.to_csv(output_path, index=False)

    print("\nInspection copy saved to:")
    print(output_path)